# DINOSim — Advanced Examples

This notebook demonstrates advanced capabilities of DINOSim:
1. Loading the dataset and basic setup
2. **Saving and Loading Precomputed Embeddings** to speed up repeated runs
3. **Saving and Loading Prompt References** for reproducibility
4. **Updating References** on the fly

Author: [@AAitorG](https://github.com/AAitorG)

## 1. Setup and Load Dataset
We will download the Drosophila VNC dataset and initialize the model as seen in the quick start notebook.

In [ ]:
import os
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from glob import glob
import torch

from napari_dinosim.utils import (
    DINOSim_pipeline,
    gaussian_kernel,
    get_img_processing_f,
    load_image,
    torch_convolve,
)

device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

# Download Dataset
print("Downloading Drosophila VNC dataset...")
repo_url = "https://github.com/unidesigner/groundtruth-drosophila-vnc.git"
target_dir = "groundtruth-drosophila-vnc"
if not os.path.exists(target_dir):
    subprocess.run(["git", "clone", repo_url])
ds_path = os.path.join(target_dir, "stack1", "raw")

all_filenames = sorted(glob(os.path.join(ds_path, "*.*.*")) + glob(os.path.join(ds_path, "*.*")))
if not all_filenames:
    raise ValueError(f"No images found in {ds_path}!")

# Load a single image for fast demonstration
idx_to_test = 0
single_img_path = all_filenames[idx_to_test]
single_image = load_image(single_img_path)
if single_image.ndim == 2:
    single_image = single_image[..., np.newaxis]
single_image_batch = single_image[np.newaxis, ...]

print("Single image shape:", single_image_batch.shape)

## 2. Initialize Model & Compute Embeddings
Features can be heavy to compute. To save time during multiple iterations, you can save the embeddings to disk.

In [ ]:
crop_shape = (512, 512)
resize_size = 518
model_size = "small"
crop_shape_c = crop_shape + (single_image_batch.shape[-1],)

print("Loading DINOv2 model...")
model = torch.hub.load("facebookresearch/dinov2", f"dinov2_vit{model_size[0]}14_reg").to(device)
model.eval()
model_dims = {'small': 384, 'base': 768, 'large': 1024, 'giant': 1536}

pipeline = DINOSim_pipeline(
    model, model.patch_size, device, 
    get_img_processing_f(resize_size), 
    feat_dim=model_dims[model_size], dino_image_size=resize_size
)

kernel = torch.tensor(gaussian_kernel(size=3, sigma=1), dtype=torch.float32, device=device)
filter_f = lambda x: torch_convolve(x, kernel)

# Compute Embeddings
print("Computing features...")
pipeline.pre_compute_embeddings(
    single_image_batch, overlap=(0, 0), padding=(0, 0), 
    crop_shape=crop_shape_c, batch_size=1, verbose=False
)

print("Done!")

### 2.1 Saving and Loading Embeddings
Use `.save_embeddings(filepath)` to persist precomputed data and `.load_embeddings(filepath)` to load it back. This can be helpful for heavy datasets or when precomputing on a powerful machine and then running analysis and visualization on a less powerful one.

In [ ]:
embeddings_path = "test_embeddings.pt"

# Save computed embeddings
print(f"Saving embeddings to {embeddings_path}")
pipeline.save_embeddings(embeddings_path)

# Load them back
print("Loading embeddings...")
pipeline.load_embeddings(embeddings_path)
print("Embeddings loaded successfully!")

## 3. Saving & Loading References
Just like embeddings, you can save your reference vectors to file and load them later. This is useful for reusing specific target segmentations.

In [ ]:
prompt_points = [
    (0, 175, 250),
    (0, 410, 350),
]

# Set the reference based on our prompts
pipeline.set_reference_vector(list_coords=prompt_points)

ref_path = "test_reference.pt"

# Save reference
print(f"Saving reference to {ref_path}")
pipeline.save_reference(ref_path)

# Clear and load reference
pipeline.delete_references()
print("Loading reference...")
pipeline.load_reference(ref_path)

# Evaluate to confirm
distances = pipeline.get_ds_distances_sameRef(verbose=False)
predictions = pipeline.distance_post_processing(distances, filter_f, upsampling_mode="bilinear")

### Visualize the results

In [ ]:
threshold = 0.5

# Visualise the prompt points on the image
xs = [p[1] for p in prompt_points]
ys = [p[2] for p in prompt_points]
img_prompt = single_image_batch[0]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(img_prompt[..., 0] if img_prompt.shape[-1] == 1 else img_prompt, cmap="gray")
axes[0].scatter(xs, ys, c="red", s=80, marker="x", linewidths=2)
axes[0].set_title("Input + Prompts")

axes[1].imshow(1 - predictions[0, ..., 0], cmap="magma")
axes[1].set_title("Similarity Map (bright = similar)")

axes[2].imshow(predictions[0, ..., 0] < threshold, cmap="gray")
axes[2].set_title(f"Segmentation Mask (threshold={threshold})")

for ax in axes: ax.axis("off")
plt.tight_layout()
plt.show()

## 4. Updating References
If you want to look for something else, simply set new reference vectors! This automatically clears the previous references without having to recompute embeddings.

In [ ]:
new_prompts = [
    (0, 330, 310),  # Different location
]

# Updating the reference seamlessly clears the old vectors
pipeline.set_reference_vector(list_coords=new_prompts)
print("References updated.")

new_distances = pipeline.get_ds_distances_sameRef(verbose=False)
new_predictions = pipeline.distance_post_processing(new_distances, filter_f, upsampling_mode="bilinear")

### Visualize the results

In [ ]:
new_xs = [p[1] for p in new_prompts]
new_ys = [p[2] for p in new_prompts]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Original Reference
axes[0, 0].imshow(img_prompt[..., 0] if img_prompt.shape[-1] == 1 else img_prompt, cmap="gray")
axes[0, 0].scatter(xs, ys, c="red", s=80, marker="x", linewidths=2)
axes[0, 0].set_title("Original Prompts")

axes[0, 1].imshow(1 - predictions[0, ..., 0], cmap="magma")
axes[0, 1].set_title("Original Similarity Map")

axes[0, 2].imshow(predictions[0, ..., 0] < threshold, cmap="gray")
axes[0, 2].set_title("Original Segmentation Mask")

# Updated Reference
axes[1, 0].imshow(img_prompt[..., 0] if img_prompt.shape[-1] == 1 else img_prompt, cmap="gray")
axes[1, 0].scatter(new_xs, new_ys, c="red", s=80, marker="x", linewidths=2)
axes[1, 0].set_title("Updated Prompts")

axes[1, 1].imshow(1 - new_predictions[0, ..., 0], cmap="magma")
axes[1, 1].set_title("Updated Similarity Map")

axes[1, 2].imshow(new_predictions[0, ..., 0] < threshold, cmap="gray")
axes[1, 2].set_title("Updated Segmentation Mask")

for ax in axes.flatten(): ax.axis("off")
plt.tight_layout()
plt.show()